In [ ]:
import os, json, re
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from shapely.geometry import shape
from PIL import Image
from multiprocessing import Pool, cpu_count

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

import multiprocessing
multiprocessing.set_start_method("fork", force=True)


In [ ]:
# fire_data_in_range = fire_data[(fire_data['date'] >= '2017-01-01') & (fire_data['date'] <= '2019-01-01')]
# print(f"Fires between 2017 and 2019: {len(fire_data_in_range)}")

In [ ]:
start_date, end_date = "2012-01-01", "2022-01-01"

env_data = pd.read_parquet("combined_ButteCounty_Averages_NEW.parquet", engine='pyarrow')
fire_data = pd.read_csv("butte_fires_NEW.csv")


env_data['date'] = pd.to_datetime(env_data['week_start'])
fire_data['date'] = pd.to_datetime(fire_data['Ig_Date'])

fire_data['date'] = fire_data['date'] - pd.to_timedelta(fire_data['date'].dt.weekday, unit='D')
env_data['date'] = env_data['date'] - pd.to_timedelta(env_data['date'].dt.weekday, unit='D')


env_data = env_data[env_data['date'] >= pd.to_datetime(start_date)]
fire_data = fire_data[fire_data['date'] >= pd.to_datetime(start_date)]
env_data = env_data[env_data['date'] <= pd.to_datetime(end_date)]
fire_data = fire_data[fire_data['date'] <= pd.to_datetime(end_date)]



fire_data = fire_data[['date', 'BurnBndAc', 'BurnBndLat', 'BurnBndLon']].copy()
fire_data['BurnBndAc'] = fire_data['BurnBndAc'].fillna(0)


grid_origin_lon = -122.5
grid_origin_lat = 39.2
chunk_size = 0.015  # 0.015 grid size


fire_data['grid_x'] = ((fire_data['BurnBndLon'] - grid_origin_lon) / chunk_size).apply(np.floor).astype(int)
fire_data['grid_y'] = ((fire_data['BurnBndLat'] - grid_origin_lat) / chunk_size).apply(np.floor).astype(int)


def get_affected_cells(row):    
    affected = []
    center_x = int(np.floor((row['BurnBndLon'] - grid_origin_lon) / chunk_size))
    center_y = int(np.floor((row['BurnBndLat'] - grid_origin_lat) / chunk_size))
    radius = int(np.ceil(np.sqrt(row['BurnBndAc']) / np.sqrt(640)))  # Approximate radius of burn area
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            affected.append((row['date'], center_x + dx, center_y + dy))
    return affected

overlay_fire_data = fire_data.apply(get_affected_cells, axis=1).explode().dropna()
overlay_fire_data = pd.DataFrame(overlay_fire_data.tolist(), columns=['date', 'grid_x', 'grid_y'])
overlay_fire_data['fire_occurred'] = 1
fire_occurrence = overlay_fire_data.drop_duplicates().reset_index(drop=True)

print("Fire occurrences:", len(fire_occurrence))
print(fire_occurrence.head())


fire_occurrence['grid_x'] = fire_occurrence['grid_x'].astype(int)
fire_occurrence['grid_y'] = fire_occurrence['grid_y'].astype(int)


env_data['grid_x'] = ((env_data['grid_x'] - grid_origin_lon) / chunk_size).apply(np.floor).astype(int) 
env_data['grid_y'] = ((env_data['grid_y'] - grid_origin_lat) / chunk_size).apply(np.floor).astype(int)


min_lat = grid_origin_lat
max_lat = grid_origin_lat + chunk_size * env_data['grid_y'].max()
min_lon = grid_origin_lon
max_lon = grid_origin_lon + chunk_size * env_data['grid_x'].max()

fire_data = fire_data[
    (fire_data['BurnBndLat'] >= min_lat) &
    (fire_data['BurnBndLat'] <= max_lat) &
    (fire_data['BurnBndLon'] >= min_lon) &
    (fire_data['BurnBndLon'] <= max_lon)
]

print(f"Fires remaining after spatial filtering: {len(fire_data)}")

data = pd.merge(env_data, fire_occurrence[['date', 'grid_x', 'grid_y', 'fire_occurred']], on=['date', 'grid_x', 'grid_y'], how='left')

data['fire_occurred'] = data['fire_occurred'].fillna(0)


percentages = data['fire_occurred'].value_counts(normalize=True) * 100
print("Fire Occurrence Distribution (%):")
print(percentages)


features = [
    "dewpoint_temperature_2m",
    "evaporation_from_bare_soil_sum",
    "volumetric_soil_water_layer_2",
    "temperature_2m",
    "total_precipitation_sum",
    "leaf_area_index_low_vegetation_min"
]

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data[features])

Extract NDVI  ### NO NEED TO RUN AGAIN

import os
from datetime import datetime

def find_ndvi_files_in_range(ndvi_root, start_date, end_date):
    ndvi_files = []

    # Convert start/end to datetime
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    for subdir in os.listdir(ndvi_root):
        subdir_path = os.path.join(ndvi_root, subdir)
        if os.path.isdir(subdir_path):
            # Look for .tif files in the subdir
            for file in os.listdir(subdir_path):
                if file.endswith(".tif"):
                    try:
                        file_date = datetime.strptime(file.replace(".tif", ""), "%Y-%m-%d")
                        if start <= file_date <= end:
                            ndvi_files.append(os.path.join(subdir_path, file))
                    except ValueError:
                        pass  # Skip if filename isn't a valid date
    print(ndvi_files)
    return ndvi_files

ndvi_files = find_ndvi_files_in_range("final_ndvi_data", start_date, end_date)





import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine
from tqdm import tqdm  # for progress bar

def resample_ndvi_to_1mile(input_path, output_path):
    with rasterio.open(input_path) as src:
        scale_factor = 1609 / 1000  # 1000m to ~1609m (1 mile)
        new_width = int(src.width / scale_factor)
        new_height = int(src.height / scale_factor)

        transform = src.transform * Affine.scale(src.width / new_width, src.height / new_height)
        
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.bilinear
        )

        profile = src.profile
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": transform
        })

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(data)

def batch_resample_folder(input_root, output_root):
    all_tifs = []
    for dirpath, _, filenames in os.walk(input_root):
        for filename in filenames:
            if filename.endswith(".tif"):
                input_path = os.path.join(dirpath, filename)
                rel_path = os.path.relpath(input_path, input_root)
                output_path = os.path.join(output_root, rel_path)
                all_tifs.append((input_path, output_path))

    print(f"🔄 Starting resampling of {len(all_tifs)} NDVI files...")
    for input_path, output_path in tqdm(all_tifs, desc="Resampling", unit="file"):
        try:
            resample_ndvi_to_1mile(input_path, output_path)
        except Exception as e:
            print(f"❌ Failed: {input_path} -> {e}")

# Example usage:
batch_resample_folder("final_ndvi_data", "resampled_ndvi_1mile")


GRIDDING NDVI ## HERE YOU CAN START RUN AGAIN

In [ ]:
from tqdm import tqdm 
def extract_ndvi_grid(tif_path):
    with rasterio.open(tif_path) as src:
        ndvi = src.read(1)
        ndvi = np.where(ndvi == src.nodata, np.nan, ndvi)

        transform = src.transform
        rows, cols = np.indices(ndvi.shape)
        xs, ys = rasterio.transform.xy(transform, rows, cols)
        
        xs = np.array(xs).flatten()
        ys = np.array(ys).flatten()
        ndvi_vals = ndvi.flatten()

        df = pd.DataFrame({
            "x": xs,
            "y": ys,
            "ndvi": ndvi_vals
        }).dropna()
        
        # Add date from filename
        filename = os.path.basename(tif_path)
        date_str = os.path.splitext(filename)[0]  # assumes filename is like 2018-04-10.tif
        df["date"] = date_str
        return df

def process_folder(input_folder, output_csv):
    all_dfs = []
    for dirpath, _, filenames in os.walk(input_folder):
        for filename in tqdm(filenames, desc="Gridding NDVI", unit="file"):
            if filename.endswith(".tif"):
                tif_path = os.path.join(dirpath, filename)
                try:
                    df = extract_ndvi_grid(tif_path)
                    all_dfs.append(df)
                except Exception as e:
                    print(f"❌ Error processing {filename}: {e}")

    full_df = pd.concat(all_dfs, ignore_index=True)
    full_df.to_csv(output_csv, index=False)
    print(f"✅ Saved gridded NDVI data to {output_csv}")

# Example usage:
process_folder("resampled_ndvi_1mile", "gridded_ndvi.csv")

INDEX THE GRID

In [ ]:
import pandas as pd

gridded_ndvi = pd.read_csv("gridded_ndvi.csv")
# Create a unique grid_id by combining x and y coordinates
# Set the minimum x and y as the origin (0, 0)
# Use the same grid origin to ensure alignment
grid_origin_lon = -122.5
grid_origin_lat = 39.2
resolution = 1609.34  # 1 mile = 1609.34 meters

# Ensure that the grid origin is correctly applied to x and y values
gridded_ndvi['row'] = ((gridded_ndvi['y'] - grid_origin_lat) / resolution).apply(np.floor).astype(int)
gridded_ndvi['col'] = ((gridded_ndvi['x'] - grid_origin_lon) / resolution).apply(np.floor).astype(int)

# If negative values appear, you might need to adjust the origin or shift them
gridded_ndvi['row'] = gridded_ndvi['row'] - gridded_ndvi['row'].min()  # Shift rows to start from 0
gridded_ndvi['col'] = gridded_ndvi['col'] - gridded_ndvi['col'].min()  # Shift columns to start from 0

# Create the grid_id in 'row_column' format
gridded_ndvi['grid_id'] = gridded_ndvi['row'].astype(str) + '_' + gridded_ndvi['col'].astype(str)




In [ ]:
# Convert the date column to datetime if it's not already in that format
gridded_ndvi['date'] = pd.to_datetime(gridded_ndvi['date'])

# Filter the dataset to include only rows within the desired date range
filtered_ndvi = gridded_ndvi[(gridded_ndvi['date'] >= start_date) & (gridded_ndvi['date'] <= end_date)]

filtered_ndvi = filtered_ndvi.sort_values(by='date')

# Check the filtered data
print(filtered_ndvi[['x', 'y', 'date', 'grid_id']])


MERGE NUMERICAL AND NDVI

In [ ]:
# Convert date columns to datetime
data['date'] = pd.to_datetime(data['date'])
filtered_ndvi['date'] = pd.to_datetime(filtered_ndvi['date'])

# Align both datasets to the start of the week (Monday)
data['week_start'] = data['date'] - pd.to_timedelta(data['date'].dt.weekday, unit='D')

# Create 'week_start' in 'filtered_ndvi' based on 'date'
filtered_ndvi['week_start'] = filtered_ndvi['date'] - pd.to_timedelta(filtered_ndvi['date'].dt.weekday, unit='D')


In [ ]:
print("FILTERED NDVI",filtered_ndvi['week_start'].unique())
print("NUM DATAA", data['week_start'].unique())


In [ ]:
tqdm.pandas()

# Assuming `data` and `filtered_ndvi` are already defined

# Perform the merge and track progress
merged_data = pd.merge(data, filtered_ndvi, on=['week_start', 'grid_id'], how='inner')

# If necessary, handle missing NDVI values (fill or drop)
# merged_data['ndvi'] = merged_data['ndvi'].fillna(0)  # Or use another method, like dropping

# Check the result
print(merged_data)


In [ ]:
# print(env_data['date'].unique()) 

# print(fire_data['date'].unique()) 

LANDSAT TIME (this is gonna be tough)

EXTRACT LANDSAT INTO DICTIONARY

In [ ]:
def extract_acquisition_date(folder_name):
    try:
        parts = folder_name.split('_')
        date_str = parts[3]  # e.g., 20130409
        return datetime.strptime(date_str, "%Y%m%d").date()
    except Exception as e:
        print(f"Error parsing date from {folder_name}: {e}")
        return None

def load_all_landsat_images(base_folder):
    all_data = []

    # List folders in the base folder
    folders = [f for f in os.listdir(base_folder) if os.path.isdir(os.path.join(base_folder, f))]

    for folder in tqdm(folders, desc="Loading Landsat folders"):
        folder_path = os.path.join(base_folder, folder)

        acquisition_date = extract_acquisition_date(folder)
        if acquisition_date is None:
            continue

        try:
            bands = {}
            # Check for each band
            for band_code in ["SR_B4", "SR_B5", "ST_B10"]:
                file_name = f"{folder}_{band_code}.png"
                file_path = os.path.join(folder_path, file_name)

                # Try loading the file directly
                if os.path.exists(file_path):
                    with Image.open(file_path) as img:
                        bands[band_code] = np.array(img)
                else:
                    # If the exact file doesn't exist, try looking for files with (1), (2), ... suffixes
                    for i in range(1, 10):  # Check for up to 9 possible suffixes (SR_B4 (1).png, SR_B4 (2).png, etc.)
                        file_path_with_suffix = f"{file_path[:-4]} ({i}){file_path[-4:]}"
                        if os.path.exists(file_path_with_suffix):
                            print(f"Found file with suffix: {file_path_with_suffix}")
                            with Image.open(file_path_with_suffix) as img:
                                bands[band_code] = np.array(img)
                            break

            # Add the data only if all bands are found
            if len(bands) == 3:
                all_data.append({
                    "date": acquisition_date,
                    "folder": folder,
                    "SR_B4": bands["SR_B4"],
                    "SR_B5": bands["SR_B5"],
                    "ST_B10": bands["ST_B10"]
                })
            else:
                print(f"Skipping folder {folder} due to missing bands.")

        except Exception as e:
            print(f"Error loading images from {folder_path}: {e}")

    return all_data


# Example usage
landsat_data = load_all_landsat_images("processed_landsat_data")
print(f"Loaded {len(landsat_data)} image sets.")
print("First entry info:")
if len(landsat_data) > 0:
    print("Date:", landsat_data[0]['date'])
    print("Band shapes:", {k: v.shape for k, v in landsat_data[0].items() if isinstance(v, np.ndarray)})

In [ ]:
# print(landsat_data[1])

GRIDDING ALL LANDSAT

In [ ]:
def grid_band_array(band_array, pixels_per_mile=54):
    h, w = band_array.shape
    rows = h // pixels_per_mile
    cols = w // pixels_per_mile
    grid = []

    for i in range(rows):
        for j in range(cols):
            window = band_array[
                i * pixels_per_mile : (i + 1) * pixels_per_mile,
                j * pixels_per_mile : (j + 1) * pixels_per_mile
            ]
            mean_val = np.nanmean(window)
            grid.append((i, j, mean_val))
    
    return pd.DataFrame(grid, columns=["row", "col", "mean"])

def grid_all_bands(landsat_data, band_name, pixels_per_mile=54):
    all_dfs = []

    for scene in tqdm(landsat_data, desc=f"Gridding {band_name}"):
        if band_name not in scene:
            print(f"⚠️ {band_name} missing in scene {scene.get('folder', 'unknown')}")
            continue
        
        band_array = scene[band_name]
        date = scene['date']
        df = grid_band_array(band_array, pixels_per_mile)
        df["date"] = date
        all_dfs.append(df)

    if all_dfs:
        full_df = pd.concat(all_dfs, ignore_index=True)
        full_df.to_csv(f"gridded_{band_name}.csv", index=False)
        print(f"✅ Saved gridded data for {band_name} to gridded_{band_name}.csv")
    else:
        print(f"❌ No data to save for {band_name}")

# Example usage for all 3 bands:
for band in ["SR_B4", "SR_B5", "ST_B10"]:
    grid_all_bands(landsat_data, band)

MERGING ALL DATA

In [ ]:
landsat_b4 = pd.read_csv("gridded_SR_B4.csv")
landsat_b5 = pd.read_csv("gridded_SR_B5.csv")
landsat_b10 = pd.read_csv("gridded_ST_B10.csv")

# Filter the dataset to include only rows wit               hin the desired date range
filtered_b4 = landsat_b4[(landsat_b4['date'] >= start_date) & (landsat_b4['date'] <= end_date)]
filtered_b5 = landsat_b5[(landsat_b5['date'] >= start_date) & (landsat_b5['date'] <= end_date)]
filtered_b10 = landsat_b10[(landsat_b10['date'] >= start_date) & (landsat_b10['date'] <= end_date)]

filtered_b4['date'] = pd.to_datetime(filtered_b4['date'])
filtered_b5['date'] = pd.to_datetime(filtered_b5['date'])
filtered_b10['date'] = pd.to_datetime(filtered_b10['date'])

filtered_b4['week_start'] = filtered_b4['date'] - pd.to_timedelta(filtered_b4['date'].dt.weekday, unit='D')
filtered_b5['week_start'] = filtered_b5['date'] - pd.to_timedelta(filtered_b5['date'].dt.weekday, unit='D')
filtered_b10['week_start'] = filtered_b10['date'] - pd.to_timedelta(filtered_b10['date'].dt.weekday, unit='D')


In [ ]:
filtered_b4['mean_b4'] = filtered_b4['mean']
filtered_b5['mean_b5'] = filtered_b5['mean']
filtered_b10['mean_b10'] = filtered_b10['mean']

filtered_b4['grid_id'] = filtered_b4['row'].astype(str) + "_" + filtered_b4['col'].astype(str)
filtered_b5['grid_id'] = filtered_b5['row'].astype(str) + "_" + filtered_b5['col'].astype(str)
filtered_b10['grid_id'] = filtered_b10['row'].astype(str) + "_" + filtered_b10['col'].astype(str)

final_merged_data = pd.merge(filtered_b4[['week_start', 'grid_id', 'mean_b4']], filtered_b5[['week_start', 'grid_id', 'mean_b5']], on=['week_start', 'grid_id'], how='inner')
final_merged_data = pd.merge(final_merged_data, filtered_b10[['week_start', 'grid_id', 'mean_b10']], on=['week_start', 'grid_id'], how='inner')


final_merged_data = pd.merge(final_merged_data, merged_data, on=['week_start', 'grid_id'], how='inner')

# # Checking the week_start values in filtered_b4
# print(filtered_b4['week_start'].unique())

# Check the final merged data
print(final_merged_data.head())

START LSTM CODE

In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

2025-05-07 13:42:03.146124: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
start_real, end_real = "2017-01-01", "2019-01-01"

filtered_data = pd.read_csv("final_data.csv")

filtered_data = filtered_data[(filtered_data['week_start'] >= start_real) & (filtered_data['week_start'] <= end_real)]

# Sort by week_start to ensure the time series is in order
filtered_data = filtered_data.sort_values(by=['grid_id', 'week_start'])
y = filtered_data['fire_occurred'].values

scaler = MinMaxScaler()
X = filtered_data[['mean_b4', 'mean_b5', 'mean_b10', 'ndvi', 'dewpoint_temperature_2m', 'evaporation_from_bare_soil_sum',
                             'temperature_2m', 'total_precipitation_sum', 'volumetric_soil_water_layer_2']].values

In [ ]:
# final_merged_data[['week_start', 'grid_id', 'mean_b4', 'mean_b5', 'mean_b10', 'ndvi', 'dewpoint_temperature_2m', 
#                              'evaporation_from_bare_soil_sum', 'temperature_2m', 'total_precipitation_sum', 'volumetric_soil_water_layer_2', 
#                              'fire_occurred']].to_csv('final_data.csv')

full = pd.read_csv("final_data.csv", index_col=0)

print("FULL columns:", full.columns.tolist())
print("fire_occurred value counts:\n", full["fire_occurred"].value_counts(dropna=False))
print("Sample with fire_occurred == 1:\n", full[full["fire_occurred"] == 1].head())

DEAL WITH MISSING DATA

In [ ]:
from sklearn.impute import SimpleImputer

import joblib

imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X)

joblib.dump(imputer, 'imputer.joblib')

In [ ]:
# # #######
# # #### Trying one month in advance (comment out for one week)
# # #######

# filtered_data['fire_next_month'] = filtered_data.groupby('grid_id')['fire_occurred'].shift(-4)
# filtered_data = filtered_data.dropna(subset=['fire_next_month'])
# y = filtered_data['fire_next_month'].astype(int).values

# ##########
# ########
# #####


DEAL WITH OVERSAMPLING

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.model_selection import StratifiedShuffleSplit


X_scaled = scaler.fit_transform(X_train_imputed)

# 4. Split before SMOTE to preserve test distribution
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(X_scaled, y):
    X_train_raw, y_train_raw = X_scaled[train_idx], y[train_idx]
    X_test, y_test = X_scaled[test_idx], y[test_idx]

# 5. Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_raw, y_train_raw)
# X_test_resampled, y_test_resampled = smote.fit_resample(X_test_raw, y_test_raw)

# 6. Create time series sequences
sequence_length = 10  # how many time steps to look back

train_generator = TimeseriesGenerator(X_train_resampled, y_train_resampled,
                                      length=sequence_length, batch_size=32)

# Optional: Create test generator if you want to evaluate with generator
test_generator = TimeseriesGenerator(X_test, y_test,
                                     length=sequence_length, batch_size=32)



In [ ]:

# # Split the data into train and test sets
# X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# # Reshape the input data for LSTM
# X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
# X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
# print("FILTERED NDVI",filtered_data['fire_occurred'].unique())
print(np.bincount(y_train_resampled.astype(int)))  # [50000 50000] or similar
# print(np.bincount(y_test_resampled))   # [12500 12500] or similar


In [ ]:
# 7. Define and compile the LSTM model
model = Sequential()
model.add(LSTM(50, activation='relu', return_sequences=True,
               input_shape=(sequence_length, X_train_resampled.shape[1])))
model.add(Dropout(0.2))
model.add(LSTM(50, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))  # Binary classification

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 8. Train the model
model.fit(train_generator, epochs=1, validation_data=test_generator, verbose=1)



In [ ]:
# # Evaluate the model
# test_loss, test_accuracy = model.evaluate(X_test, y_test)
# print(f"Test accuracy: {test_accuracy}")

# # Predictions on test data
# predictions = model.predict(X_test)
# print(predictions)
    

# model.save('my_model.keras')

from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split

# # Suppose these are your outputs on validation data (after training)
# raw_preds = model.predict(X_test).flatten()
# true_labels = y_test.flatten()

# # Fit isotonic regression
# calibrator = IsotonicRegression(out_of_bounds='clip')
# calibrator.fit(raw_preds, true_labels)

# 9. Evaluate and predict
y_pred_prob = model.predict(test_generator)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Evaluate metrics
from sklearn.metrics import classification_report, confusion_matrix

y_test_trimmed = y_test[sequence_length:]  # align with TimeseriesGenerator output
print(confusion_matrix(y_test_trimmed, y_pred))
print(classification_report(y_test_trimmed, y_pred, zero_division=0))

In [ ]:

# # Calculate the percentage of predictions equal to 1
# percentage_ones = (predictions >= 0.5).mean() * 100

# print(f"Percentage of predictions equal to 1: {percentage_ones:.2f}%")

# print(fire_proneness_scores)

# import joblib
# joblib.dump(calibrator, 'calibrator.pkl')

# print(raw_preds)

# model.summary()

# print("Input shape to model:", sequence_array.shape)
# print("Sample input (first sequence):")
# print(sequence_array[0])

from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

y_val_probs = model.predict(train_generator).flatten()

# 3. Get the matching labels (aligned with generator output)
# Generator skips the first `sequence_length` samples, so trim y_train_resampled accordingly
y_calibration_labels = y_train_resampled[sequence_length:]

# 4. Reshape probabilities for calibration input
X_cal = y_val_probs.reshape(-1, 1)

# 5. Calibrate using logistic regression
calibrator = LogisticRegression()
calibrator.fit(X_cal, y_calibration_labels)

WEB APP PREDICTION

In [3]:
# Load the dataset
df = pd.read_csv("final_data.csv")
df_sorted = df.sort_values(["grid_id", "week_start"])

feature_cols =  ['mean_b4', 'mean_b5', 'mean_b10', 'ndvi', 'dewpoint_temperature_2m', 'evaporation_from_bare_soil_sum',
                             'temperature_2m', 'total_precipitation_sum', 'volumetric_soil_water_layer_2']



X_full_imputed = imputer.fit_transform(df_sorted[feature_cols])
X_full_scaled = scaler.fit_transform(X_full_imputed)

# Add scaled data back to the DataFrame
df_sorted[feature_cols] = X_full_scaled

# 2. Build sequences per grid
sequence_length = 10
X_sequences = []
grid_ids = []

for grid_id, group in df_sorted.groupby('grid_id'):
    if len(group) >= sequence_length + 1:
        last_10 = group.iloc[-(sequence_length+1):-1]  # final 10 weeks, excluding last
        X_seq = last_10[feature_cols].values
        X_sequences.append(X_seq)
        grid_ids.append(grid_id)

X_sequences = np.array(X_sequences)  

print(X_sequences)


NameError: name 'imputer' is not defined

In [7]:
df = pd.read_csv("map/fire-proneness-map/predictions_v2.csv")
df_sorted = df.sort_values(["grid_id", "week_start"])

print(df['week_start'].nunique())
print(df['grid_id'].nunique())
print(df.shape)

291
4221
(1198165, 8)


In [ ]:
raw_probs = model.predict(X_sequences).flatten()

# 4. Calibrate
# Fit calibrator on training data first (you must have already done this during training!)
# Example:
# y_train_probs = model.predict(train_generator).flatten()
# calibrator = LogisticRegression()
# calibrator.fit(y_train_probs.reshape(-1, 1), y_train_resampled)

# Now calibrate
calibrated_probs = calibrator.predict_proba(raw_probs.reshape(-1, 1))[:, 1]

# 5. Create DataFrame of predictions
predictions_df = pd.DataFrame({
    'grid_id': grid_ids,
    'calibrated_prob': calibrated_probs,
    'raw_prob': raw_probs
})



In [ ]:
scaler_new = MinMaxScaler()
scaled_predictions = calibrated_probs.flatten().reshape(-1, 1)
scaled_predictions = scaler_new.fit_transform(scaled_predictions)

predictions_df['scaled'] = scaled_predictions/10
# Apply log-squashing (add small value to avoid log(0))
predictions_df['log_scaled_prob'] = np.log1p(calibrated_probs)

# or square root (gentler)
predictions_df['sqrt_scaled_prob'] = np.power((calibrated_probs),(0.33))

from scipy.special import softmax
predictions_df['softmax_prob'] = softmax(predictions_df['calibrated_prob'].values)
print(predictions_df)

predictions_df.to_csv('newest_week_predictions.csv')

5 YEAR PREDICTION START

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load data
df = pd.read_csv("final_data.csv")
df_sorted = df.sort_values(["grid_id", "week_start"])

feature_cols = ['mean_b4', 'mean_b5', 'mean_b10', 'ndvi',
                'dewpoint_temperature_2m', 'evaporation_from_bare_soil_sum',
                'temperature_2m', 'total_precipitation_sum', 'volumetric_soil_water_layer_2']

# Impute and scale
X_full_imputed = imputer.fit_transform(df_sorted[feature_cols])
X_full_scaled = scaler.fit_transform(X_full_imputed)
df_sorted[feature_cols] = X_full_scaled

sequence_length = 10
all_predictions = []


NameError: name 'imputer' is not defined

In [ ]:
from tqdm import tqdm 
X_all = []
grid_ids_all = []
week_starts_all = []

print("Generating sequences...")
for grid_id, group in tqdm(df_sorted.groupby("grid_id")):
    group = group.reset_index(drop=True)
    for i in range(len(group) - sequence_length):
        window = group.iloc[i:i + sequence_length]
        X_seq = window[feature_cols].values
        X_all.append(X_seq)
        grid_ids_all.append(grid_id)
        week_starts_all.append(group.iloc[i + sequence_length]["week_start"])

X_all = np.array(X_all)  # shape: (samples, 10, features)


In [ ]:
# 3. Predict
print("Running model predictions...")
raw_probs = model.predict(X_all).flatten()

# 4. Calibrate
print("Calibrating predictions...")
calibrated_probs = calibrator.predict_proba(raw_probs.reshape(-1, 1))[:, 1]

# 5. Create DataFrame
print("Creating predictions DataFrame...")
predictions_df = pd.DataFrame({
    'grid_id': grid_ids_all,
    'week_start': week_starts_all,
    'raw_prob': raw_probs,
    'calibrated_prob': calibrated_probs
})

print(predictions_df)

In [ ]:
from scipy.special import softmax
# 6. Scale and transform
scaler_new = MinMaxScaler()
scaled_predictions = scaler_new.fit_transform(calibrated_probs.reshape(-1, 1))
predictions_df['scaled'] = scaled_predictions / 10
predictions_df['log_scaled_prob'] = np.log1p(calibrated_probs)
predictions_df['sqrt_scaled_prob'] = np.power(calibrated_probs, 0.33)
predictions_df['softmax_prob'] = softmax(predictions_df['calibrated_prob'].values)

# 7. Save to CSV
predictions_df.to_csv("all_predictions_5years.csv", index=False)
print("✅ Saved predictions to all_predictions_5years.csv")

////  END WEB APP PREDICTION

TRYING ONE MONTH IN ADVANCE

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler

scaler = MinMaxScaler()
X = filtered_data[['mean_b4', 'mean_b5', 'mean_b10', 'ndvi', 'dewpoint_temperature_2m', 'evaporation_from_bare_soil_sum',
                             'temperature_2m', 'total_precipitation_sum', 'volumetric_soil_water_layer_2']].values


# ros = RandomOverSampler(sampling_strategy='auto', random_state=42)
# X_oversampled, y_oversampled = ros.fit_resample(X, y)

# # Now, apply SMOTE to generate synthetic samples from the minority class
# smote = SMOTE(sampling_strategy='auto', random_state=42)
# X_resampled, y_resampled = smote.fit_resample(X_oversampled, y_oversampled)


scaled_data = scaler.fit_transform(X)

# # Prepare the LSTM data: We need to create time windows for each grid_id
# # Using TimeseriesGenerator to create the time series sequences
# sequence_length = 10  # Use 10 weeks of data to predict the next week (adjust as needed)

# # Create a generator to handle time series data for each grid_id
# generator = TimeseriesGenerator(X_resampled, y_resampled, 
#                                 length=sequence_length, batch_size=32)          

In [ ]:

# --------------------------------------------
# 2. Sequence Generator with Forecast Horizon
# --------------------------------------------

def create_sequences(features, labels, seq_length, forecast_horizon):
    X, y = [], []
    for i in range(len(features) - seq_length - forecast_horizon):
        X.append(features[i:i+seq_length])
        # Label is 1 if any fire occurs in the forecast window
        y.append(int(np.any(labels[i+seq_length:i+seq_length+forecast_horizon])))
    return np.array(X), np.array(y)

seq_length = 30           # Past 30 weeks
forecast_horizon = 4      # Predict if fire will occur in next 4 weeks (1 month)
X, y = create_sequences(scaled_data, y, seq_length, forecast_horizon)

In [ ]:
# --------------------------------------------
# 3. Train-Test Split + Oversampling
# --------------------------------------------

X_reshaped = X.reshape(X.shape[0], -1)  # Temporarily flatten for SMOTE
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Check the number of samples in the resampled data
print(f"X_resampled shape: {X_resampled.shape}")
print(f"y_resampled shape: {len(y_resampled)}")

# Ensure the number of samples is divisible by the sequence length
assert X_resampled.shape[0] % seq_length == 0, "Number of samples is not divisible by seq_length"

# Reshape the resampled X into a 3D array (samples, sequence_length, features)
X_resampled_reshaped = X_resampled.reshape(-1, seq_length, len(features))

# Ensure that X_resampled_reshaped and y_resampled have the same number of samples
assert X_resampled_reshaped.shape[0] == len(y_resampled), "Number of samples mismatch!"


# Check if the reshaped X_resampled and y_resampled have the same number of samples
print(f"X_resampled_reshaped shape: {X_resampled_reshaped.shape}")
print(f"y_resampled shape: {len(y_resampled)}")
